In [ ]:
!pip install transformers evaluate accelerate

In [ ]:
from datasets import load_dataset

raw_datasets = load_dataset("eriktks/conll2003", revision="convert/parquet")

In [ ]:
raw_datasets

In [ ]:
raw_datasets['train'][0]

In [ ]:
raw_datasets['train'][0]['tokens']

In [ ]:
ner_features = raw_datasets['train'].features['ner_tags']

In [ ]:
ner_features

In [ ]:
label_names = ner_features.feature.names

In [ ]:
label_names

In [ ]:
words = raw_datasets["train"][0]["tokens"]
labels = raw_datasets["train"][0]["ner_tags"]
line1 = ""
line2 = ""

for word,label in zip(words,labels):
  full_label=label_names[label]
  max_length= max(len(word),len(full_label))
  line1 += word + " " * (max_length - len(word) + 1)
  line2 += full_label + " "*(max_length-len(full_label)+1)

print(line1)
print(line2)

In [ ]:
raw_datasets

In [ ]:
pos_labels = raw_datasets['train'].features['pos_tags'].feature.names
chunk_labels = raw_datasets['train'].features['chunk_tags'].feature.names

In [ ]:
# now lets see everything for index 4
print(raw_datasets['train'][4]['tokens'])

In [ ]:
index_4_tokens = raw_datasets['train'][4]['tokens']
index_4_ners = raw_datasets['train'][4]['ner_tags']
index_4_poss = raw_datasets['train'][4]['pos_tags']
index_4_chunks = raw_datasets['train'][4]['chunk_tags']

print(f"{'TOKEN':<15} {'NER_TAG':<15} {'POS_TAG':<15} {'CHUNK_TAG'}")

for t, n, p, c in zip(index_4_tokens, index_4_ners, index_4_poss, index_4_chunks):
    ner_name = label_names[n]
    pos_name = pos_labels[p]
    chunk_name = chunk_labels[c]

    print(f"{t:<15} {ner_name:<15} {pos_name:<15} {chunk_name}")

In [ ]:
from transformers import AutoTokenizer
checkpoint="bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)



In [ ]:
tokenizer.is_fast

In [ ]:
input = tokenizer(raw_datasets['train'][0]['tokens'],is_split_into_words=True)

In [ ]:
input

In [ ]:
print(f"{"type":>15} {"length":>15}")
print(f"{"token length":>15} {len(raw_datasets['train'][0]['tokens']):>15}")
print(f"{"input id":>15} {len(input['input_ids']):>15}")
print(f"{"token_type_ids":>15} {len(input['token_type_ids']):>15}")
print(f"{"attention_masks":>15} {len(input['attention_mask']):>15}")

In [ ]:
input.tokens()

In [ ]:
input.word_ids()

In [ ]:
## we have great problem , called token label alignment problem , as shape of tokens and labels i.e ner tags must match
##, but we will do tokenization then
## certain words will be broken down again into subwords , so what ner tags they should be provided

In [ ]:
def align_labels_with_tokens(labels,word_ids):
  new_labels=[]
  current_word=None
  for word_id in word_ids:
    if word_id != current_word:
      current_word = word_id
      label = -100 if word_id is None else labels[word_id]
      new_labels.append(label)
    elif word_id is None:
      new_labels.append(-100)
    else:
      label=labels[word_id]

      if label% 2 == 1:
        label += 1
      new_labels.append(label)
  return new_labels

In [ ]:
def tokenize_and_align_labels(examples):
  tokenized_inputs = tokenizer(examples["tokens"],truncation=True, is_split_into_words=True)
  all_labels = examples["ner_tags"]
  new_labels=[]
  for i,label in enumerate(all_labels):
    word_ids = tokenized_inputs.word_ids(i)
    new_labels.append(align_labels_with_tokens(label,word_ids))

  tokenized_inputs["labels"]=new_labels
  return tokenized_inputs




In [ ]:
tokenized_datasets = raw_datasets.map(tokenize_and_align_labels,
                                      batched=True,
                                      remove_columns=raw_datasets["train"].column_names)

In [ ]:
from transformers import DataCollatorForTokenClassification
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

In [ ]:
!pip install seqeval

In [ ]:
import evaluate
metric = evaluate.load('seqeval')

In [ ]:
import numpy as np

def compute_metrics(eval_preds):

    logits, labels = eval_preds

    # Correct axis
    predictions = np.argmax(logits, axis=-1)

    true_labels = [
        [label_names[l] for l in label if l != -100]
        for label in labels
    ]

    true_predictions = [
        [label_names[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    all_metrics = metric.compute(
        predictions=true_predictions,
        references=true_labels
    )

    return {
        "precision": all_metrics["overall_precision"],
        "recall": all_metrics["overall_recall"],
        "f1": all_metrics["overall_f1"],
        "accuracy": all_metrics["overall_accuracy"],
    }

In [ ]:
id2label = {i:label for i,label in enumerate(label_names)}
label2id = {v:k for k,v in id2label.items()}

In [ ]:
from transformers import AutoModelForTokenClassification
model_checkpoint=checkpoint
model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    id2label=id2label,
    label2id=label2id
)

In [ ]:
model.config.num_labels

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from transformers import TrainingArguments

args = TrainingArguments(
    "bert-finetuned-ner",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    num_train_epochs=3,
    weight_decay=0.01,
    push_to_hub=True,
)

In [ ]:
from transformers import Trainer
trainer = Trainer(
    model=model,
    args=args,
    train_dataset = tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator = data_collator,
    compute_metrics = compute_metrics,
    processing_class = tokenizer,
)
trainer.train()

In [ ]:
trainer.push_to_hub(commit_message="Training complete")